# 02 — Table 1: who are the births in the study?

**Goal.** Describe the study population: the share of births in each category of the main variables, among survivors and among infants who died, and how these shares change from 2016 to 2023.

**Thesis part.** Step 1. This becomes "Table 1" of the thesis. It also shows how much data is missing for each variable.

**Inputs.** `data/parquet/cohort_<year>.parquet`.

**Outputs.** `reports/table1_by_outcome.csv`, `reports/table1_by_cohort.csv`.

**How to run.** Run all cells (about 1 minute).

**Terms.**
- *Survivor*: a baby who did not die before age one. *Death*: a linked infant death before day 365.
- *Column %*: the share of a group (for example, of all deaths) that falls in one category.
- *Unknown*: the certificate has no value for this item. We show it as its own row.

## 1. Setup

In [1]:
from pathlib import Path

import duckdb
import polars as pl
ROOT = Path.cwd().parent            # notebooks/ -> InfantMortalityThesis/
pl.Config.set_tbl_rows(60); pl.Config.set_tbl_cols(20); pl.Config.set_fmt_str_lengths(40)
PARQUET = ROOT / "data" / "parquet"
COHORTS = range(2016, 2024)

### Value labels

**What.** The plain-English label of each code, for the variables in Table 1. The codes come from the user guides (`reference/columns.csv`). `DPLURAL = 5` is not in any guide; it appears only in 2016–2019, so we show it as its own row.

In [2]:
YN = {"Y": "Yes", "N": "No", "U": "Unknown"}

TABLE1_VARS = {
    "MAGER9": ("Mother's age", {"1": "<15", "2": "15-19", "3": "20-24", "4": "25-29", "5": "30-34",
                                "6": "35-39", "7": "40-44", "8": "45-49", "9": "50-54"}),
    "MRACEHISP": ("Mother's race/Hispanic origin", {"1": "NH White", "2": "NH Black", "3": "NH AIAN",
                                                     "4": "NH Asian", "5": "NH NHOPI", "6": "NH more than one race",
                                                     "7": "Hispanic", "8": "Unknown"}),
    "MBSTATE_REC": ("Mother's nativity", {"1": "Born in the U.S.", "2": "Born outside the U.S.", "3": "Unknown"}),
    "MEDUC": ("Mother's education", {"1": "8th grade or less", "2": "9-12th, no diploma", "3": "High school/GED",
                                      "4": "Some college", "5": "Associate", "6": "Bachelor's", "7": "Master's",
                                      "8": "Doctorate/professional", "9": "Unknown"}),
    "PAY_REC": ("Payment source", {"1": "Medicaid", "2": "Private insurance", "3": "Self-pay", "4": "Other", "9": "Unknown"}),
    "LBO_REC": ("Live birth order", {"1": "1st", "2": "2nd", "3": "3rd", "4": "4th", "5": "5th", "6": "6th",
                                     "7": "7th", "8": "8th or more", "9": "Unknown"}),
    "CIG0_R": ("Cigarettes per day before pregnancy", {"0": "None", "1": "1-5", "2": "6-10", "3": "11-20",
                                                        "4": "21-40", "5": "41+", "6": "Unknown"}),
    "BMI_R": ("Pre-pregnancy BMI", {"1": "Underweight <18.5", "2": "Normal 18.5-24.9", "3": "Overweight 25.0-29.9",
                                    "4": "Obesity I 30.0-34.9", "5": "Obesity II 35.0-39.9",
                                    "6": "Obesity III >=40", "9": "Unknown"}),
    "RF_PDIAB": ("Pre-pregnancy diabetes", YN),
    "RF_GDIAB": ("Gestational diabetes", YN),
    "RF_PHYPE": ("Pre-pregnancy hypertension", YN),
    "RF_GHYPE": ("Gestational hypertension", YN),
    "DPLURAL": ("Plurality", {"1": "Single", "2": "Twin", "3": "Triplet", "4": "Quadruplet+", "5": "Code 5 (not in guide; 2016-2019 only)"}),
    "SEX": ("Infant sex", {"M": "Male", "F": "Female"}),
}

## 2. Table 1 by outcome (all cohorts, U.S. residents)

**What.** For each variable, count births in each category, separately for survivors and deaths, and compute column percentages.

**Why.** Readers must see the population before any model. Large differences between survivors and deaths show which variables will matter. This is a description, not an adjusted result.

**Expected.** Deaths are more often twins, babies of Black mothers, Medicaid births and births to mothers with pre-pregnancy hypertension or diabetes.

In [3]:
ALL = f"'{PARQUET}/cohort_*.parquet'"

def table1(by: str) -> pl.DataFrame:
    out = []
    for var, (title, labels) in TABLE1_VARS.items():
        q = f"""SELECT {by} AS grp, COALESCE({var}, 'missing') AS code, COUNT(*) AS n
                FROM read_parquet({ALL}) WHERE CAST(RESTATUS AS INT) < 4 GROUP BY ALL"""
        d = pl.from_arrow(duckdb.sql(q).arrow()).with_columns(
            pl.col("grp").cast(pl.Utf8),
            pct=(100 * pl.col("n") / pl.col("n").sum().over("grp")).round(2),
            variable=pl.lit(title),
            category=pl.col("code").replace_strict(labels, default=pl.col("code")),
        )
        out.append(d)
    return pl.concat(out)

by_outcome = table1("CASE WHEN death THEN 'Death' ELSE 'Survivor' END")
wide = by_outcome.pivot(on="grp", index=["variable", "code", "category"], values=["n", "pct"]).sort("variable", "code")
wide = wide.select("variable", "code", "category", "n_Survivor", "n_Death", "pct_Survivor", "pct_Death")
wide.write_csv(ROOT / "reports" / "table1_by_outcome.csv")
wide

variable,code,category,n_Survivor,n_Death,pct_Survivor,pct_Death
str,str,str,i64,i64,f64,f64
"""Cigarettes per day before pregnancy""","""0""","""None""",27512045,142775,92.58,85.82
"""Cigarettes per day before pregnancy""","""1""","""1-5""",555660,5696,1.87,3.42
"""Cigarettes per day before pregnancy""","""2""","""6-10""",696192,6977,2.34,4.19
"""Cigarettes per day before pregnancy""","""3""","""11-20""",693628,6848,2.33,4.12
"""Cigarettes per day before pregnancy""","""4""","""21-40""",105011,1154,0.35,0.69
"""Cigarettes per day before pregnancy""","""5""","""41+""",21565,236,0.07,0.14
"""Cigarettes per day before pregnancy""","""6""","""Unknown""",131873,2681,0.44,1.61
"""Gestational diabetes""","""N""","""No""",27520388,157399,92.61,94.61
"""Gestational diabetes""","""U""","""Unknown""",34541,742,0.12,0.45


**Result.** The table gives counts (`n_…`) and column percentages (`pct_…`) for survivors and deaths. A category that is much more common among deaths is linked to higher risk *before* any adjustment. These crude differences are not effects; Part 2 adjusts for confounders.

As expected, deaths are more often twins (12.2% vs 3.2% of survivors), births to NH Black mothers (27.5% vs 14.2%), Medicaid births (54.1% vs 41.6%), and births to mothers with pre-pregnancy hypertension (4.5% vs 2.4%) or diabetes (2.2% vs 1.0%). Obesity classes II–III are also more common among deaths.

**Three findings that change later steps:**
1. **Missing values are more common among deaths.** BMI is unknown for 6.4% of deaths but 2.3% of survivors; education for 4.5% vs 1.5%. This is far above the plan's 20% trigger (section 4, check 5), so Part 2 **must** add the multiple-imputation sensitivity analysis. A complete-case analysis alone could be biased.
2. **Gestational diabetes looks "protective" (4.9% of deaths vs 7.3% of survivors).** The likely reason is timing: it is usually tested at 24–28 weeks, so many very preterm babies are born before the test. This is the leakage risk named in the plan. In Part 1, compare Set A (without it) and Set B (with it), and read its importance with care.
3. **Plurality has an undocumented code 5** (126 births, 2016–2019). It is probably quintuplets or more; it is shown as its own row.

## 3. Table 1 by cohort year

**What.** The same percentages for all births, one column per cohort.

**Why.** Trends over time matter for Part 1, where we train on 2016–2020 and test on 2022–2023. A variable whose distribution shifts a lot can make an older model less accurate in later years.

**Expected.** Rising pre-pregnancy hypertension, gestational diabetes and obesity; falling smoking before pregnancy.

In [4]:
by_cohort = table1("cohort")
wide_c = by_cohort.pivot(on="grp", index=["variable", "code", "category"], values="pct").sort("variable", "code")
wide_c = wide_c.select("variable", "code", "category", *[str(c) for c in COHORTS])
wide_c.write_csv(ROOT / "reports" / "table1_by_cohort.csv")
wide_c

variable,code,category,2016,2017,2018,2019,2020,2021,2022,2023
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""Cigarettes per day before pregnancy""","""0""","""None""",90.13,90.57,91.14,91.83,92.55,93.71,94.98,95.86
"""Cigarettes per day before pregnancy""","""1""","""1-5""",2.44,2.34,2.21,2.04,1.92,1.6,1.28,1.1
"""Cigarettes per day before pregnancy""","""2""","""6-10""",3.21,3.01,2.82,2.59,2.36,1.95,1.51,1.23
"""Cigarettes per day before pregnancy""","""3""","""11-20""",3.18,3.05,2.84,2.56,2.32,1.94,1.51,1.19
"""Cigarettes per day before pregnancy""","""4""","""21-40""",0.47,0.45,0.43,0.38,0.36,0.3,0.24,0.19
"""Cigarettes per day before pregnancy""","""5""","""41+""",0.08,0.08,0.08,0.08,0.08,0.07,0.06,0.05
"""Cigarettes per day before pregnancy""","""6""","""Unknown""",0.49,0.49,0.48,0.51,0.4,0.43,0.42,0.38
"""Gestational diabetes""","""N""","""No""",93.95,93.56,93.26,93.01,92.09,91.57,91.74,91.59
"""Gestational diabetes""","""U""","""Unknown""",0.1,0.1,0.08,0.09,0.11,0.14,0.18,0.16


**Result.** Each number is the percentage of all resident births in that year. Read across a row to see the trend.

The expected trends are present: pre-pregnancy hypertension rises from 1.7% (2016) to 3.2% (2023), gestational hypertension from 6.0% to 10.1%, gestational diabetes from 6.0% to 8.3%, and obesity class III from 4.8% to 6.2%. Smoking before pregnancy falls (no smoking: 90.1% to 95.9%). Payment source and plurality change little.

These shifts are large for the exposures, so the 2022–2023 test years differ from the 2016–2020 training years. Part 1 must report calibration per test year.

## Summary

- **Main finding.** The tables describe 29.9 million resident births and 166 thousand infant deaths, 2016–2023. The crude differences between deaths and survivors have the expected directions.
- **Open issues.** (1) Missing BMI and education are about 3 times more common among deaths, so multiple imputation is required in Part 2. (2) Gestational diabetes has a timing problem for prediction. (3) The exposures rise over time, so Part 1 must report calibration per test year.
- **Next notebook.** `03_prediction.ipynb` (Part 1).